In [3]:
# EDA

import duckdb
import pandas as pd
import time
"""
C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_train.parquet
C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_test.parquet
"""
# 파일 경로 지정
parquet_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_train.parquet"

# 판다스 출력 제한 해제 (모든 컬럼과 행을 숨김없이 표시)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(f"[{parquet_file}]")
print("종합 EDA 및 데이터 무결성 검증을 시작합니다 ...\n")
start_time = time.time()

# DuckDB 인메모리 연결
con = duckdb.connect()

try:
    # ---------------------------------------------------------
    # 1. 데이터 규격 (행/열 개수)
    # ---------------------------------------------------------
    print("=== [1. 데이터 규격 확인] ===")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{parquet_file}')").fetchone()[0]
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    col_names = schema_df['column_name'].tolist()
    
    print(f"총 행 수(Rows): {total_rows:,} 개")
    print(f"총 열 수(Columns): {len(col_names)} 개\n")

    # ---------------------------------------------------------
    # 2. 상위 10개 데이터 샘플 (모든 열 표시)
    # ---------------------------------------------------------
    print("=== [2. 상위 10개 데이터 샘플 (생략 없음)] ===")
    sample_df = con.execute(f"SELECT * FROM read_parquet('{parquet_file}') LIMIT 10").fetchdf()
    print(sample_df)
    print("\n")

    # ---------------------------------------------------------
    # 3. 하드디스크 개체 및 클래스 분포 통계 (ML Target 반영 수정 완료)
    # ---------------------------------------------------------
    print("=== [3. 하드디스크 개체 및 클래스 분포 통계] ===")
    status_query = f"""
        SELECT 
            COUNT(DISTINCT serial_number) AS total_objects,
            COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number ELSE NULL END) AS failed_objects,
            COUNT(*) AS total_rows,
            SUM(CAST(failure AS INTEGER)) AS target_1_rows,
            SUM(CASE WHEN failure = 0 THEN 1 ELSE 0 END) AS target_0_rows
        FROM read_parquet('{parquet_file}')
    """
    stats = con.execute(status_query).fetchdf().iloc[0]

    total_obj = stats['total_objects']
    failed_obj = stats['failed_objects']
    healthy_obj = total_obj - failed_obj
    
    print("[개체 단위 통계 (물리적인 하드디스크 개수)]")
    print(f"- 전체 고유 개체 수: {int(total_obj):,} 개")
    print(f"- 정상 작동 하드: {int(healthy_obj):,} 개 ({(healthy_obj/total_obj)*100:.2f}%)")
    print(f"- 고장 발생 개체: {int(failed_obj):,} 개 ({(failed_obj/total_obj)*100:.2f}%)")
    print(f"- 개체 단위 비율 (Class 1 : 0) = 1 : {healthy_obj / failed_obj:.2f}\n")

    print("[행 단위 클래스 분포 (❗진짜 ML 모델이 학습할 Target 레이블 비율)]")
    total_r = stats['total_rows']
    target_1 = stats['target_1_rows']
    target_0 = stats['target_0_rows']
    
    print(f"- 총 데이터 행 수: {int(total_r):,} 개")
    print(f"- Class 0 (정상인 날): {int(target_0):,} 개 ({target_0/total_r*100:.2f}%)")
    print(f"- Class 1 (고장 임박): {int(target_1):,} 개 ({target_1/total_r*100:.2f}%)")
    
    if target_1 > 0:
        ratio = target_0 / target_1
        print(f"- 실제 타겟 데이터 불균형 비율 (Class 1 : 0) = 1 : {ratio:.1f}\n")

    # ... [1~3번 섹션까지는 기존 코드와 동일] ...

    # ---------------------------------------------------------
    # 5. 열 별 간단한 기초 통계 (Numeric Data) - 안전 모드
    # ---------------------------------------------------------
    print("=== [5. 열 별 간단한 기초 통계 (Numeric Data)] ===")
    
    # 숫자형 컬럼 리스트 추출 (DESCRIBE 활용)
    numeric_types = ['BIGINT', 'DOUBLE', 'FLOAT', 'INTEGER', 'HUGEINT', 'UTINYINT']
    num_cols = schema_df[schema_df['column_type'].isin(numeric_types)]['column_name'].tolist()
    
    stats_list = []
    
    print(f"총 {len(num_cols)}개 숫자형 컬럼 통계 계산 중...")
    
    for col in num_cols:
        try:
            # CAST(col AS DOUBLE)을 통해 계산 가능 범위를 최대로 확보합니다.
            query = f"""
                SELECT 
                    MIN("{col}") as min_val,
                    MAX("{col}") as max_val,
                    AVG(CAST("{col}" AS DOUBLE)) as avg_val,
                    STDDEV_SAMP(CAST("{col}" AS DOUBLE)) as std_val
                FROM read_parquet('{parquet_file}')
            """
            res = con.execute(query).fetchone()
            stats_list.append({
                "column_name": col,
                "min": res[0],
                "max": res[1],
                "avg": round(res[2], 2) if res[2] is not None else None,
                "std": round(res[3], 2) if res[3] is not None else None
            })
        except Exception as e:
            # STDDEV_SAMP 에러 발생 시 std만 제외하고 기록하거나 Overflow 메시지 표시
            # 에러가 나더라도 루프를 멈추지 않고 계속 진행합니다.
            stats_list.append({
                "column_name": col,
                "min": "Error",
                "max": "Error",
                "avg": "Error",
                "std": "Overflow/Error"
            })
            print(f"  ⚠️ '{col}' 컬럼 계산 건너뜀 (수치 범위 초과)")

    # 결과 출력
    stats_df = pd.DataFrame(stats_list)
    pd.set_option('display.max_rows', None)
    print(stats_df)
    pd.reset_option('display.max_rows')
    print("\n")

except Exception as e:
    print(f"❌ 검증 중 치명적 오류 발생: {e}")
finally:
    con.close()
    # ... [이하 동일]


<>:6: SyntaxWarning: invalid escape sequence '\W'
<>:6: SyntaxWarning: invalid escape sequence '\W'
C:\Users\joon6\AppData\Local\Temp\ipykernel_21092\2012463929.py:6: SyntaxWarning: invalid escape sequence '\W'
  """


[C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_train.parquet]
종합 EDA 및 데이터 무결성 검증을 시작합니다 ...

=== [1. 데이터 규격 확인] ===
총 행 수(Rows): 298,815 개
총 열 수(Columns): 310 개

=== [2. 상위 10개 데이터 샘플 (생략 없음)] ===
  serial_number       date  failure  smart_5_raw  smart_184_raw  smart_187_raw  smart_197_raw  smart_198_raw  timeout_5s  timeout_total  seek_error_count  smart_9_raw  smart_189_raw  smart_191_raw  smart_194_raw  smart_199_raw  smart_241_raw  smart_242_raw  total_reads  total_seeks  smart_183_raw  smart_190_raw  s5_diff  s184_diff  s187_diff  s197_diff  s198_diff  timeout_5s_diff  timeout_total_diff  seek_error_count_diff  s189_diff  s191_diff  s194_diff  s199_diff  s241_diff   s242_diff  total_reads_diff  total_seeks_diff  s183_diff  s190_diff  age_weighted_seek_error  fatal_crash_interaction  shock_seek_interaction  age_weighted_workload  late_stage_degradation  cumulative_error_score  firmware_struggle_index  log_shock_fly_interaction  error_growth_ratio  io_asymme

In [ ]:
import duckdb

# 1. 다시 연결
con = duckdb.connect()

# 2. 경로 재설정 (필요한 경우)
parquet_file = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data\rfe_sample_train.parquet"

# 3. 이상치 조사 쿼리 실행
query = f"""
    SELECT 
        serial_number, date, 
        timeout_total_diff, timeout_5s_diff,
        timeout_severity_ratio
    FROM read_parquet('{parquet_file}')
    ORDER BY timeout_severity_ratio DESC
    LIMIT 10
"""
extreme_values = con.execute(query).df()
print("=== [timeout_severity_ratio 극단값 TOP 10] ===")
print(extreme_values)

# 조사가 끝나면 닫기 (또는 주석 처리)
# con.close()


ConnectionException: Connection Error: Connection already closed!